In [ ]:
import os
import sys
import json
import glob
import pickle
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/naver/disk2/HoangDPB/rPPG-Toolbox"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.BigSmall import BigSmall

In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "raw_data")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_bigsmall_nb")
MODEL_PATH          = os.path.join(REPO_ROOT, "final_model_release", "BP4D_BigSmall_Multitask_Fold1.pth") # 1 or 2 or 3
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results_bigsmall")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 25       # PPG sensor sampling rate (Hz)

# ----- BigSmall preprocessing params -----
CHUNK_LENGTH = 3       # frames per clip (same as training config)
BIG_H, BIG_W = 144, 144
SMALL_H, SMALL_W = 9, 9

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----- 49-channel label layout (matches BigSmallTrainer label_list) -----
LABEL_IDX_BVP  = 0   # bp_wave (green-channel PPG signal)
LABEL_IDX_HR   = 1   # HR_bpm
LABEL_IDX_RESP = 5   # resp_wave
NUM_LABEL_CH   = 49

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [ ]:
# Read and time-align PPG signal

def read_ppg_synced(session_path, num_frames, fps=30):
    """Read PPG green channel from ppg.csv and resample to video frame times.

    The PPG sensor timestamps and video start timestamp are both in ms
    Unix epoch stored in metadata.json (sync_markers.video_start).
    Resampling is done by linear interpolation onto frame-spaced timestamps.

    Returns:
        ppg (np.ndarray): shape (num_frames,), dtype float32, raw green values.
    """
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start")
    if video_start_ms is None:
        video_start_ms = meta.get("start_timestamp")
    if video_start_ms is None:
        raise ValueError(f"No video_start or start_timestamp found in {meta_path}")
    video_start_ms = float(video_start_ms)

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)

    # convert ppg timestamps to seconds relative to video start
    ppg_t = (ppg_ts - video_start_ms) / 1000.0

    # frame timestamps in seconds
    frame_t = np.arange(num_frames, dtype=np.float64) / fps

    # clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])

    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_green)
    return ppg_resampled.astype(np.float32)

In [ ]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [ ]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Resize all frames (no face detection)

def resize_frames(frames, out_h, out_w):
    """Resize every frame to (out_h, out_w) using INTER_AREA."""
    T, H, W, C = frames.shape
    resized = np.zeros((T, out_h, out_w, C), dtype=np.float32)
    for i in range(T):
        resized[i] = cv2.resize(frames[i], (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects + read device-measured HR from hr.csv

def read_gt_hr(session_path):
    """Return average HR from hr.csv (hrStatus==1, hr>0).

    hr.csv has IBI columns whose values are arrays like [720, 695],
    which contain commas and break the standard CSV parser.
    Reading only the first 5 columns avoids the issue.
    """
    hr_path = os.path.join(session_path, "hr.csv")
    df = pd.read_csv(
        hr_path,
        usecols=[0, 1, 2, 3, 4],
        names=["id", "dataReceived", "timestamp", "hr", "hrStatus"],
        header=0,
    )
    valid = df[(df["hrStatus"] == 1) & (df["hr"] > 0)]["hr"]
    if len(valid) == 0:
        return float("nan")
    return float(valid.mean())


# Discover subject folders directly — no metadata.csv required.
# avg_hr is set to 0.0 (placeholder); it only fills an unused label channel
# and has no effect on any reported metric.
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_dirs = glob.glob(os.path.join(RAW_DATA_PATH, subj_id, "*"))
    if not session_dirs:
        print(f"No session folder for {subj_id}, skipping.")
        continue
    session_path = session_dirs[0]

    session_name  = os.path.basename(session_path)
    video_pattern = os.path.join(RAW_DATA_PATH, "videos",
                                  f"{subj_id}_{session_name}.mp4")
    video_files = glob.glob(video_pattern)
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    gt_hr = read_gt_hr(session_path)

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
        "avg_hr":       0.0,   # placeholder — not used in any reported metric
        "gt_hr":        gt_hr,
    })
    print(f"  {subj_id}  HR_device={gt_hr:.4f} bpm  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

In [ ]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]
    avg_hr       = subj["avg_hr"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T, fps=VIDEO_FPS)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # 49-channel label array; most channels stay -1
    labels = np.full((T, NUM_LABEL_CH), -1.0, dtype=np.float32)
    labels[:, LABEL_IDX_BVP] = ppg_signal
    labels[:, LABEL_IDX_HR]  = avg_hr

    frames_big = crop_face_resize(frames, BIG_H, BIG_W)

    big_data   = standardized_data(frames_big)
    diff_big   = diff_normalize_data(frames_big)
    small_data = resize_frames(diff_big, SMALL_H, SMALL_W)

    labels[:, LABEL_IDX_BVP] = diff_normalize_label(ppg_signal)

    clip_num    = T // CHUNK_LENGTH
    big_clips   = np.array([big_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]   for i in range(clip_num)])
    small_clips = np.array([small_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([labels[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]     for i in range(clip_num)])

    # subject-specific subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.pickle")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        frames_dict = {0: big_clips[chunk_idx], 1: small_clips[chunk_idx]}
        with open(input_path, "wb") as fh:
            pickle.dump(frames_dict, fh, protocol=pickle.HIGHEST_PROTOCOL)
        np.save(label_path, label_clips[chunk_idx])
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*.pickle")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

In [ ]:
# PyTorch Dataset

class BigSmallDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label").replace(".pickle", ".npy")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        with open(self.inputs[index], "rb") as fh:
            data = pickle.load(fh)

        # NDHWC -> NDCHW
        data[0] = np.float32(np.transpose(data[0], (0, 3, 1, 2)))
        data[1] = np.float32(np.transpose(data[1], (0, 3, 1, 2)))

        label = np.float32(np.load(self.labels[index]))  # (D, 49)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = BigSmallDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

In [ ]:
# Custom collate function + DataLoader

def bigsmall_collate(batch):
    """Stack dict-based data into tensors.

    Returns:
        data_big   (Tensor): (N, D, C, H_big, W_big)
        data_small (Tensor): (N, D, C, H_small, W_small)
        labels     (Tensor): (N, D, 49)
        subjects   (list[str])
        chunk_ids  (list[str])
    """
    data_dicts, labels, subjects, chunk_ids = zip(*batch)

    data_big   = torch.stack([torch.from_numpy(d[0]) for d in data_dicts], dim=0)
    data_small = torch.stack([torch.from_numpy(d[1]) for d in data_dicts], dim=0)
    labels_t   = torch.stack([torch.from_numpy(l)    for l in labels],     dim=0)

    return data_big, data_small, labels_t, list(subjects), list(chunk_ids)


loader = DataLoader(dataset, batch_size=32, shuffle=False,
                    num_workers=4, collate_fn=bigsmall_collate)
print(f"DataLoader ready: {len(loader)} batches")

In [ ]:
# Load pretrained BigSmall model

model = BigSmall(n_segment=CHUNK_LENGTH)

state_dict = torch.load(MODEL_PATH, map_location=DEVICE)

# strip 'module.' prefix if present (DataParallel artifact)
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print("Model loaded and set to eval mode.")
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

In [ ]:
# Inference 

BASE_LEN = CHUNK_LENGTH  # num_gpu=1

# BVP
bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
bvp_labels_dict = {}

# RESP
resp_preds_dict = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}

with torch.no_grad():
    for batch in tqdm(loader, desc="Inference"):
        data_big, data_small, labels_t, batch_subjects, batch_chunk_ids = batch

        N, D, C_b, H_b, W_b = data_big.shape
        N, D, C_s, H_s, W_s = data_small.shape

        # flatten temporal dimension
        big_flat   = data_big.view(N * D, C_b, H_b, W_b).to(DEVICE)
        small_flat = data_small.view(N * D, C_s, H_s, W_s).to(DEVICE)

        # trim to multiple of BASE_LEN for TSM alignment
        trim = (N * D) // BASE_LEN * BASE_LEN
        big_flat   = big_flat[:trim]
        small_flat = small_flat[:trim]

        labels_flat = labels_t.view(N * D, NUM_LABEL_CH)  # (N*D, 49)

        au_out, bvp_out, resp_out = model((big_flat, small_flat))

        bvp_pred_np  = bvp_out.squeeze(-1).cpu().numpy()   # (trim,)
        resp_pred_np = resp_out.squeeze(-1).cpu().numpy()  # (trim,)
        bvp_label_np = labels_flat[:trim, LABEL_IDX_BVP].numpy()

        for i in range(N):
            if i * CHUNK_LENGTH >= trim:
                break

            subj  = batch_subjects[i]
            cid   = int(batch_chunk_ids[i])
            start = i * CHUNK_LENGTH
            end   = start + CHUNK_LENGTH

            if subj not in bvp_preds_dict:
                bvp_preds_dict[subj]  = {}
                bvp_labels_dict[subj] = {}
                resp_preds_dict[subj] = {}

            bvp_preds_dict[subj][cid]  = bvp_pred_np[start:end]
            bvp_labels_dict[subj][cid] = bvp_label_np[start:end]
            resp_preds_dict[subj][cid] = resp_pred_np[start:end]

print("\nInference complete.")
print("Subjects:", sorted(bvp_preds_dict.keys()))

In [ ]:
# Post-processing

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])

# Per-subject BVP post-processing

def process_bvp(pred_chunks, label_chunks, fs=30):
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    pred  = detrend(np.cumsum(pred),  100)
    label = detrend(np.cumsum(label), 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

# Per-subject RESP post-processing

def process_resp(pred_chunks, fs=30):
    pred = _reform_from_dict(pred_chunks).astype(np.float64)

    pred = detrend(np.cumsum(pred), 100)
    pred = bandpass_filter(pred, fs, low=0.13, high=0.5)

    rr_pred_bpm = fft_peak_hz(pred, fs, 0.13, 0.5) * 60.0
    return rr_pred_bpm

In [ ]:
# Blink rate detection

def detect_blink_rate(video_path, fps=30, large_box_coef=1.5):
    """Estimate blink rate (blinks/min) using eye-strip brightness analysis.

    Steps:
      1. Detect face on frame 0 with Haar Cascade.
      2. Define eye strip = rows [20%, 50%] of face bbox height.
      3. Compute mean grayscale brightness in eye strip per frame.
      4. Invert (blinks = dips) and bandpass filter [0.1, 0.9] Hz.
      5. Count peaks with minimum separation of 1 second.
    """
    frames = read_video_frames(video_path)
    T = len(frames)

    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray0, scaleFactor=1.3, minNeighbors=5)
    H, W = frames[0].shape[:2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H

    eye_top    = y + int(fh * 0.20)
    eye_bottom = y + int(fh * 0.50)
    eye_left   = x
    eye_right  = x + fw

    brightness = np.array([
        np.mean(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)[eye_top:eye_bottom, eye_left:eye_right])
        for f in frames
    ])

    # blinks = dips in brightness -> invert for find_peaks
    brightness_inv = -brightness.astype(np.float64)

    # bandpass [0.1, 0.9] Hz = [6, 54] blinks/min
    b, a = signal.butter(2, [0.1 / fps * 2, 0.9 / fps * 2], btype="bandpass")
    filtered = signal.filtfilt(b, a, brightness_inv)

    # minimum 1 second between detected blinks
    min_dist = int(fps)
    peaks, _ = signal.find_peaks(filtered, distance=min_dist)

    duration_min = T / fps / 60.0
    return len(peaks) / duration_min


# Compute blink rate for each subject
blink_rate_dict = {}
for subj in subjects:
    subj_key   = subj["subj_key"]
    video_path = subj["video_path"]
    print(f"  {subj_key} ...", end=" ", flush=True)
    blink_rate = detect_blink_rate(video_path, fps=VIDEO_FPS)
    blink_rate_dict[subj_key] = blink_rate
    print(f"{blink_rate:.2f} blinks/min")

print("\nBlink rates:", {k: f"{v:.2f}" for k, v in blink_rate_dict.items()})

In [ ]:
# Compute per-subject results

FS = VIDEO_FPS

# lookup: subj_key -> subject info dict (contains gt_hr)
subjects_by_key = {s["subj_key"]: s for s in subjects}

# blink_rate_dict is populated by the blink detection cell;
# fall back to empty dict if that cell has not been run yet.
if "blink_rate_dict" not in dir():
    blink_rate_dict = {}

per_subject_results = []

hr_preds_all = []
gt_hrs_all   = []
snr_all      = []
rr_preds_all = []

print(f"{'Subject':<10} {'HR_pred':>10} {'HR_gt':>10} {'HR_err':>8} {'RR_pred':>8} {'SNR':>7} {'Blinks':>7}")
print("-" * 68)

for subj_key in sorted(bvp_preds_dict.keys()):
    hr_pred, hr_label, _, pred_processed = process_bvp(
        bvp_preds_dict[subj_key], bvp_labels_dict[subj_key], fs=FS
    )
    rr_pred    = process_resp(resp_preds_dict[subj_key], fs=FS)

    gt_hr      = subjects_by_key[subj_key]["gt_hr"]
    snr_db     = calculate_snr(pred_processed, gt_hr, FS)
    blink_rate = blink_rate_dict.get(subj_key, float("nan"))
    hr_err     = hr_pred - gt_hr

    # map subj_key ("S000") back to original ID ("S_000")
    subj_id = subj_key[0] + "_" + subj_key[1:]

    per_subject_results.append({
        "name":                subj_id,
        "predicted_heartrate": hr_pred,
        "real_heartrate":      gt_hr,
        "heartrate_error":     hr_err,
        "respiration_rate":    rr_pred,
        "blink_rate":          blink_rate,
        "snr_db":              snr_db,
    })

    hr_preds_all.append(hr_pred)
    gt_hrs_all.append(gt_hr)
    snr_all.append(snr_db)
    rr_preds_all.append(rr_pred)

    print(f"{subj_id:<10} {hr_pred:>10.3f} {gt_hr:>10.3f} {hr_err:>8.3f} {rr_pred:>8.3f} {snr_db:>7.2f} {blink_rate:>7.2f}")

hr_preds_all = np.array(hr_preds_all)
gt_hrs_all   = np.array(gt_hrs_all)
snr_all      = np.array(snr_all)

In [ ]:
# Aggregate metrics

n = len(hr_preds_all)
assert n > 0, "No subjects to evaluate."

err   = hr_preds_all - gt_hrs_all
abs_e = np.abs(err)
sq_e  = err ** 2
rel_e = abs_e / (np.abs(gt_hrs_all) + 1e-9)

mae       = float(np.mean(abs_e))
mae_se    = float(np.std(abs_e) / np.sqrt(n))

rmse      = float(np.sqrt(np.mean(sq_e)))
rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))

mape      = float(np.mean(rel_e) * 100.0)
mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

if n >= 2:
    pearson_r  = float(np.corrcoef(hr_preds_all, gt_hrs_all)[0, 1])
    pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
else:
    pearson_r, pearson_se = float("nan"), float("nan")

mean_snr    = float(np.mean(snr_all))
mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

print(f"MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
print(f"RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
print(f"MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
print(f"Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
print(f"SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

In [ ]:
# Export results

model_name = os.path.basename(MODEL_PATH).replace(".pth", "")

metrics_dict = {
    "model":      model_name,
    "n_subjects": n,
    "evaluation_method": "FFT",
    "bvp_bandpass_hz":   [0.6, 3.3],
    "resp_bandpass_hz":  [0.13, 0.5],
    "aggregate_metrics": {
        "MAE":     {"value": mae,      "se": mae_se,      "unit": "bpm"},
        "RMSE":    {"value": rmse,     "se": rmse_se,     "unit": "bpm"},
        "MAPE":    {"value": mape,     "se": mape_se,     "unit": "%"},
        "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
        "SNR":     {"value": mean_snr, "se": mean_snr_se, "unit": "dB"},
    },
    "per_subject": [
        {
            "name":               r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "real_heartrate":      r["real_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "respiration_rate":    r["respiration_rate"],
            "snr_db":              r["snr_db"],
        }
        for r in per_subject_results
    ],
}

json_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(json_path, "w") as fh:
    json.dump(metrics_dict, fh, indent=2)

print(f"Metrics saved to: {json_path}")

csv_rows = []
for r in per_subject_results:
    csv_rows.append({
        "name":               r["name"],
        "predicted_heartrate": r["predicted_heartrate"],
        "real_heartrate":      r["real_heartrate"],
        "heartrate_error":     r["heartrate_error"],
        "respiration_rate":    r["respiration_rate"],
        "blink_rate":          r["blink_rate"],
    })

results_df = pd.DataFrame(csv_rows, columns=[
    "name", "predicted_heartrate", "real_heartrate",
    "heartrate_error", "respiration_rate", "blink_rate"
])

csv_path = os.path.join(OUTPUT_DIR, "ppg_results.csv")
results_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")
print()
print(results_df.to_string(index=False))